## Spot rate estimation 
### Deriving spot rate from IRS rate
Coupon rate -> Discount rate

### 1. Bootstraping

Decomposition of swap contract into two bond

In [1]:
# Using Hypothetical data 
# Assuming 3M spot rate = 0
# I guess the complication about 3M spot rate is coming from the payment schedule

import pandas as pd 
import sympy as sp

df_swap_market = pd.DataFrame(
    columns = ["Maturity", "Frequency", "Swap rate", "Spot rate", "Forward rate", "Spot rate soln", "Discount soln"],
    data = [
        [0.25, 0.25, None, 0,  sp.Symbol("L3"), None, None], 
        [0.50, 0.25, 1, sp.Symbol("R6"),  sp.Symbol("L6") , None, None],
        [0.75, 0.25, 2, sp.Symbol("R9"),  sp.Symbol("L9") , None, None],
        [1.00, 0.25, 3, sp.Symbol("R12"), sp.Symbol("L12"), None, None],
    ]
)

df_swap_market["Discount"] = [
    sp.exp(-x["Spot rate"]*x["Maturity"]) for i, x in df_swap_market.iterrows()
    ]

# Assuming the floating leg cashflow is par yield bond (PV=100, Principal=100)
df_swap_market["Par yield PV"] = [
    sp.Eq(100,(x["Swap rate"] * x["Frequency"] * df_swap_market["Discount"].loc[:i]).sum() + 100 * x["Discount"]) for i, x in df_swap_market.iterrows()
    ]

print("System of equations")
for eqn in df_swap_market["Par yield PV"].iloc[1:].tolist():
    display(eqn)
    

System of equations


Eq(100, 0.25 + 100.25*exp(-0.5*R6))

Eq(100, 0.5 + 100.5*exp(-0.75*R9) + 0.5*exp(-0.5*R6))

Eq(100, 0.75 + 0.75*exp(-0.75*R9) + 0.75*exp(-0.5*R6) + 100.75*exp(-1.0*R12))

In [2]:
print("Bootstraping")
for i, x in df_swap_market.iloc[1:].iterrows():
    eqn = x["Par yield PV"]
    # Substitute known solution
    for j, y in df_swap_market.iloc[1:i].iterrows():
        eqn = eqn.subs(y["Spot rate"], y["Spot rate soln"])
    display(eqn)
    soln = sp.nsolve(eqn, x["Spot rate"], 0)
    df_swap_market.loc[i,"Spot rate soln"] = float(soln)

    # Discount factor
    df_swap_market.loc[i,"Discount soln"] = float(x["Discount"].subs(x["Spot rate"], soln))

display(df_swap_market[["Maturity", "Swap rate", "Spot rate soln", "Discount soln", "Forward rate"]])

# TODO Forward rate

Bootstraping


Eq(100, 0.25 + 100.25*exp(-0.5*R6))

Eq(100, 0.997506234413965 + 100.5*exp(-0.75*R9))

Eq(100, 2.23508393196114 + 100.75*exp(-1.0*R12))

,Maturity,Swap rate,Spot rate soln,Discount soln,Forward rate
0,0.25,NaN,None,None,L3
1,0.50,1.0,0.01,0.995012,L6
2,0.75,2.0,0.020017,0.985099,L9
3,1.00,3.0,0.030076,0.970371,L12


### 2. 

Decomposing into FRA contracts

In [3]:
# TODO Decompose into FRA, and check if the result is the same.

## About swap contract cashflow schedule

Fixing in advance 

### Cashflow schedule of fixed rate payer

* 2024-12
  * Transaction : Agreed to pay X% p.a with frequency 3M (not common), receive LIBOR 3M with frequency 3M. Maturity 1y.
  * Settlement : L3 

* 2025-03
  * Settlement : L6 
  * Payment : X/4% - L3/4$

* 2025-06
  * Settlement : L9
  * Payment : X/4% - L6/4$

* 2025-09
  * Settlement : L12%
  * Payment : X/4% - L9/4$

* 2025-09
  * Payment : X/4% - L12/4$